# SR-AttELM-H with two-time-scale online refinement: reproduction of the experiments

The notebook clones the repository, downloads the Beijing Multi-Site Air Quality data set (UCI, https://doi.org/10.24432/C5RK5G),
checks the data array and the evaluation protocol, runs a quick reproduction check (about 5 min on a Colab CPU runtime) and,
depending on the flags of the configuration cell, recomputes:

1. experiment 3, adaptation of the output layer (Tables 9–12, Fig. 5), about 1–1.5 h on a 2-core CPU runtime;
2. experiment 1, adapted reproductions of GRU-D, BRITS, SAITS and GRIN (Tables 6, 7), about 2 h;
3. experiment 2, contribution of the components (Table 8), about 30–40 min;
4. optionally, the official AGG core (several hours).

Experiments that are not recomputed are summarized from the result tables shipped with the repository.
Use *Runtime → Run all*.

In [ ]:
# Configuration
import os
REPO_URL = "https://github.com/ALAXAY/sr-attelm-h-adaptation.git"
REPO_DIR = "sr-attelm-h-adaptation"
RUN_ADAPTATION = True      # experiment 3 (Tables 9-12)
RUN_BASELINES = False      # experiment 1: GRU-D, BRITS, SAITS, GRIN (Tables 6, 7)
RUN_COMPONENTS = False     # experiment 2 (Table 8)
RUN_AGG = False            # official AGG core (several hours on a CPU runtime)
REGENERATE_DELAYS = False  # regenerate the network-derived delays instead of using the shipped realizations
THREADS = str(os.cpu_count() or 2)

In [ ]:
import pathlib, subprocess, sys

def run(cmd, cwd="."):
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError("command failed: " + " ".join(cmd))

def fresh(*names):
    """Rename shipped result tables so that the resumable scripts recompute them from the beginning."""
    for n in names:
        p = pathlib.Path("srattelm") / n
        if p.exists():
            p.replace(p.with_name(p.stem + ".shipped" + p.suffix))

if not os.path.isdir(REPO_DIR):
    run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
os.environ["SRATTELM_THREADS"] = THREADS
os.environ["AGG_THREADS"] = THREADS
os.environ["CUDA_VISIBLE_DEVICES"] = ""
print(os.getcwd(), "threads", THREADS)

In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

In [ ]:
# Data set (about 8 MB); the loader verifies the SHA-256 of the loaded array
import urllib.request
UCI_URL = "https://archive.ics.uci.edu/static/public/501/beijing+multi+site+air+quality+data.zip"
if not list(pathlib.Path("data").glob("*.zip")) and not os.environ.get("SRATTELM_DATA"):
    urllib.request.urlretrieve(UCI_URL, "data/beijing_multi_site_air_quality.zip")
sys.path.insert(0, os.path.abspath("srattelm"))
import protocol as PR
X, STATIONS = PR.load_raw()
print(X.shape, STATIONS)

In [ ]:
# Protocol check: number of delayed components and LOCF RMSE per run (geometric delays, test seeds 11, 23, 37)
import numpy as np
import run_experiments as RE
EXPECTED = {"A": ([31722, 31718, 31728], [0.5089, 0.5015, 0.5166]),
            "B": ([42042, 42044, 42038], [0.5714, 0.5749, 0.5554])}
for f, (n_exp, r_exp) in EXPECTED.items():
    _, tev = RE.views_geo(f)
    rows = RE.locf_rows(tev, f, "geometric")
    n = [int(r["n"]) for r in rows]
    rm = [r["RMSE"] for r in rows]
    assert n == n_exp and np.allclose(rm, r_exp, atol=1.5e-4), (f, n, rm)
    print("split", f, n, np.round(rm, 4), "- coincides with the evaluation protocol of the article")

In [ ]:
# Quick reproduction check: frozen output layer and two-time-scale refinement, split A, seed 11, regimes x4 and x8
run([sys.executable, "check_reproduction.py"], cwd="srattelm")

In [ ]:
# Network-derived delays: optional regeneration, statistics (Table 4, Section 7.1) and Fig. 3
if REGENERATE_DELAYS:
    run([sys.executable, "generate_delays.py"], cwd="netsim")
    run([sys.executable, "generate_training_delays.py"], cwd="netsim")
run([sys.executable, "delay_statistics.py"], cwd="netsim")
run([sys.executable, "make_figure.py"], cwd="netsim")
from IPython.display import Image, display
display(Image("netsim/out/Fig8_network_delays.png", width=900))

In [ ]:
# Experiment 3: adaptation of the output layer (Tables 9-12)
import pandas as pd
pd.set_option("display.width", 220)
if RUN_ADAPTATION:
    fresh("results_adaptation.csv", "results_adaptation2.csv", "results_adaptation3.csv")
    for script in ("adapt_experiment.py", "adapt_experiment2.py", "adapt_experiment3.py"):
        run([sys.executable, script], cwd="srattelm")
else:
    print("RUN_ADAPTATION = False: the shipped results_adaptation*.csv are used")
run([sys.executable, "summarize_adaptation.py"], cwd="srattelm")
display(pd.read_csv("srattelm/summary_adaptation.csv").round(4))

In [ ]:
# Experiment 1: comparison with existing methods (Tables 6, 7); uses the technique rows of experiment 3
if RUN_BASELINES:
    fresh("results_baselines.csv")
    run([sys.executable, "baselines.py"], cwd="srattelm")
if RUN_AGG:
    fresh("results_official_agg.csv")
    run([sys.executable, "run_agg.py"], cwd="srattelm/agg")
run([sys.executable, "summarize_methods.py"], cwd="srattelm")
display(pd.read_csv("srattelm/summary_methods.csv").round(4))

In [ ]:
# Experiment 2: contribution of the components of the technique (Table 8)
if RUN_COMPONENTS:
    run([sys.executable, "run_experiments.py"], cwd="srattelm")
    run([sys.executable, "run_component_check.py"], cwd="srattelm")
run([sys.executable, "summarize.py"], cwd="srattelm")
run([sys.executable, "summarize_components.py"], cwd="srattelm")
display(pd.read_csv("srattelm/availability_check_summary.csv"))

In [ ]:
# Comparison with the values reported in the article (pooled RMSE over six paired runs)
ARTICLE_6 = {('two_scale', 'geo_x4'): 0.2866, ('two_scale', 'geo_x8'): 0.3531, ('two_scale', 'network'): 0.3884, ('SAITS', 'geo_x4'): 0.3039, ('SAITS', 'geo_x8'): 0.4048, ('SAITS', 'network'): 0.4525, ('GRIN', 'geo_x4'): 0.4173, ('GRIN', 'geo_x8'): 0.5302, ('GRIN', 'network'): 0.5347, ('GRU-D', 'geo_x4'): 0.465, ('GRU-D', 'geo_x8'): 0.6051, ('GRU-D', 'network'): 0.6589, ('BRITS', 'geo_x4'): 0.4732, ('BRITS', 'geo_x8'): 0.635, ('BRITS', 'network'): 0.7576, ('AGG', 'geo_x4'): 0.3451, ('AGG', 'network'): 0.4974, ('LOCF', 'geo_x4'): 0.543, ('LOCF', 'geo_x8'): 0.73, ('LOCF', 'network'): 0.7968}
ARTICLE_9 = {('frozen', 'geo_x4'): 0.3305, ('frozen', 'geo_x8'): 0.431, ('frozen', 'network'): 0.4782, ('sector_stat', 'geo_x4'): 0.3192, ('sector_stat', 'geo_x8'): 0.4033, ('sector_stat', 'network'): 0.456, ('offset', 'geo_x4'): 0.3185, ('offset', 'geo_x8'): 0.4103, ('offset', 'network'): 0.4675, ('two_scale', 'geo_x4'): 0.3118, ('two_scale', 'geo_x8'): 0.3938, ('two_scale', 'network'): 0.4486}
ARTICLE_8 = {('geometric', 'proposed'): 0.2941, ('network', 'proposed'): 0.3894, ('geometric', 'weights_noav'): 0.3278, ('network', 'weights_noav'): 0.4608}
SM = pd.read_csv("srattelm/summary_methods.csv")
SA = pd.read_csv("srattelm/summary_adaptation.csv")
AV = pd.read_csv("srattelm/availability_check_summary.csv")
print("Table 6 (preliminary stage on delay levels 1, 2, 4)")
for (m, r), v in ARTICLE_6.items():
    x = float(SM[SM.method == m]["RMSE_" + r].iloc[0])
    print(f"  {m:10s} {r:8s} article {v:.4f}   this run {x:.4f}")
print("Table 9 (preliminary stage on delay level 1, geometric delays)")
for (c, r), v in ARTICLE_9.items():
    x = float(SA[(SA.family == "geometric") & (SA.coverage == "single") & (SA.regime == r)][c].iloc[0])
    print(f"  {c:12s} {r:8s} article {v:.4f}   this run {x:.4f}")
print("Table 8")
for (dl, var), v in ARTICLE_8.items():
    x = float(AV[(AV.delays == dl) & (AV.variant == var)].RMSE_pooled.iloc[0])
    print(f"  {var:12s} {dl:9s} article {v:.4f}   this run {x:.4f}")